# Spherical Linear Bayesian Optimization on 2D Branin Function

This notebook demonstrates the spherical linear regression method from the paper "We Still Don't Understand High-Dimensional Bayesian Optimization".

The method applies Bayesian linear regression after projecting the input space onto a hypersphere using inverse stereographic projection.

In [ ]:
# Install dependencies
%pip install -q botorch>=0.13 gpytorch>=1.14.3 torch>=2.1 matplotlib

In [ ]:
import math
import torch
import gpytorch
import botorch
import numpy as np
import matplotlib.pyplot as plt
from botorch.acquisition import LogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.utils import standardize
from gpytorch.constraints import GreaterThan
from gpytorch.priors import LogNormalPrior
from linear_operator.operators import (
    LowRankRootLinearOperator,
    MatmulLinearOperator,
    RootLinearOperator,
)

print(f"PyTorch version: {torch.__version__}")
print(f"BoTorch version: {botorch.__version__}")
print(f"GPyTorch version: {gpytorch.__version__}")

## Define the Branin Function

The Branin function is a common benchmark for optimization.

In [ ]:
def branin(x1, x2):
    """Branin function (to be minimized)."""
    y = float(
        (x2 - 5.1 / (4 * np.pi**2) * x1**2 + 5.0 / np.pi * x1 - 6.0) ** 2
        + 10 * (1 - 1.0 / (8 * np.pi)) * np.cos(x1)
        + 10
    )
    return y

def branin_tensor(X):
    """Branin function for tensors (to be maximized for BO, so we negate)."""
    x1 = X[..., 0]
    x2 = X[..., 1]
    y = (
        (x2 - 5.1 / (4 * np.pi**2) * x1**2 + 5.0 / np.pi * x1 - 6.0) ** 2
        + 10 * (1 - 1.0 / (8 * np.pi)) * torch.cos(x1)
        + 10
    )
    return -y  # Negate for maximization

## Implement the Spherical Linear Kernel

This kernel projects the input space onto a hypersphere using inverse stereographic projection, then applies a linear kernel.

In [ ]:
def project_onto_unit_sphere(x):
    """
    Project inputs onto sphere after scaling by lengthscale.
    Uses inverse stereographic projection.
    
    Args:
        x: Input tensor of shape (..., N, D)
    
    Returns:
        Projected tensor of shape (..., N, D+1)
    """
    x_sq_norm = x.square().sum(dim=-1, keepdim=True)
    x_ = torch.cat([2 * x, (x_sq_norm - 1.0)], dim=-1).mul(
        1.0 / (1.0 + x_sq_norm)
    )
    return x_


def maybe_low_rank_root_lo(root):
    """Create appropriate linear operator based on rank."""
    n, r = root.shape[-2:]
    if r >= n:
        return RootLinearOperator(root)
    else:
        return LowRankRootLinearOperator(root)


class SphericalLinearKernel(gpytorch.kernels.RBFKernel):
    """
    Apply linear kernel after spherical projection.
    
    This kernel:
    1. Centers and scales inputs by ARD lengthscales
    2. Applies global lengthscale
    3. Projects onto hypersphere via inverse stereographic projection
    4. Computes linear kernel (inner product) in the spherical space
    """
    
    has_lengthscale = True
    
    def __init__(self, ard_num_dims, bounds=(0.0, 1.0), batch_shape=torch.Size([])):
        if ard_num_dims == 1:
            raise ValueError(f"ard_num_dims must be >= 2. Got {ard_num_dims}.")
        
        if isinstance(bounds[0], float):
            bounds = [(bounds[0], bounds[1])] * ard_num_dims
        
        # Set up lengthscale prior (simplified version)
        lengthscale_prior = LogNormalPrior(
            loc=math.sqrt(2.0), scale=math.sqrt(3.0)
        )
        lengthscale_constraint = GreaterThan(
            2.5e-2, transform=None, initial_value=lengthscale_prior.mode
        )
        
        super().__init__(
            ard_num_dims=ard_num_dims,
            batch_shape=batch_shape,
            lengthscale_prior=lengthscale_prior,
            lengthscale_constraint=lengthscale_constraint,
        )
        
        # Create buffer for the center and length of each dimension
        _dtype = self.raw_lengthscale.dtype
        _bounds = torch.tensor(bounds, dtype=_dtype)
        self.register_buffer("_mins", _bounds[..., 0])
        self.register_buffer("_maxs", _bounds[..., 1])
        self.register_buffer("_centers", (self._mins + self._maxs).div(2.0))
        
        # Learnable coefficients for constant and linear terms
        coeffs = torch.zeros(2, dtype=_dtype)
        self.register_parameter("raw_coeffs", torch.nn.Parameter(coeffs))
        
        # Global lengthscale
        glob_ls = torch.zeros(1, dtype=_dtype)
        self.register_parameter("raw_glob_ls", torch.nn.Parameter(glob_ls))
    
    @property
    def coeffs(self):
        """The coefficients for the constant and linear terms."""
        return torch.nn.functional.softmax(self.raw_coeffs, dim=-1)
    
    @property
    def glob_ls(self):
        """The global lengthscale."""
        return torch.sigmoid(self.raw_glob_ls)
    
    def forward(self, x1, x2, diag=False, **params):
        x1_equal_x2 = torch.equal(x1, x2)
        
        # Get constants
        lengthscale = self.lengthscale
        max_sq_norm = (
            (self._maxs - self._mins)[..., None, :]
            .div(2.0 * lengthscale)
            .square()
            .sum(dim=-1, keepdim=True)
        )
        glob_ls = torch.sqrt(self.glob_ls * max_sq_norm)
        
        # Center and scale inputs
        x1 = x1.sub(self._centers).div(lengthscale)
        x2 = x1 if x1_equal_x2 else x2.sub(self._centers).div(lengthscale)
        
        # Apply global lengthscale
        x1 = x1.div(glob_ls)
        x2 = x2.div(glob_ls)
        
        # Project the inputs onto the sphere
        x1_ = project_onto_unit_sphere(x1)
        x2_ = project_onto_unit_sphere(x2)
        
        # Sum up the (weighted) components for constant and linear terms
        terms = self.coeffs
        term0_sqrt = terms[0].sqrt()
        term1_sqrt = terms[1].sqrt()
        x1_ = torch.cat([x1_ * term1_sqrt, term0_sqrt.expand_as(x1_[..., :1])], dim=-1)
        
        if x1_equal_x2:
            kernel = maybe_low_rank_root_lo(x1_)
        else:
            x2_ = torch.cat([x2_ * term1_sqrt, term0_sqrt.expand_as(x2_[..., :1])], dim=-1)
            kernel = MatmulLinearOperator(x1_, x2_.mT)
        
        return kernel

## Initialize the GP Model with Spherical Linear Kernel

In [ ]:
def initialize_model(X_train, Y_train):
    """
    Initialize the GP model with spherical linear kernel.
    
    Args:
        X_train: Training inputs of shape (n, d)
        Y_train: Training outputs of shape (n, 1)
    
    Returns:
        BoTorch SingleTaskGP model
    """
    d = X_train.size(-1)
    
    mean = gpytorch.means.ConstantMean()
    kernel = SphericalLinearKernel(ard_num_dims=d, bounds=(-5.0, 10.0) if d == 2 else (0.0, 1.0))
    
    likelihood = gpytorch.likelihoods.GaussianLikelihood(
        noise_prior := LogNormalPrior(loc=-4.0, scale=1.0),
        noise_constraint=GreaterThan(1e-4, initial_value=noise_prior.mode),
    )
    
    return botorch.models.SingleTaskGP(
        train_X=X_train,
        train_Y=Y_train,
        mean_module=mean,
        covar_module=kernel,
        likelihood=likelihood,
    )


def fit_model(model):
    """
    Fit the GP model to the training data.
    
    Args:
        model: BoTorch GP model
    """
    model.train()
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(
        likelihood=model.likelihood, model=model
    )
    botorch.fit.fit_gpytorch_mll(mll)

## Run Bayesian Optimization Loop

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Define bounds (Branin function bounds)
bounds_lower = torch.tensor([-5.0, 0.0], dtype=torch.float64)
bounds_upper = torch.tensor([10.0, 10.0], dtype=torch.float64)
bounds = torch.stack([bounds_lower, bounds_upper])

# Settings
n_init = 5  # Initial random samples
n_iterations = 15  # BO iterations
n_tot = n_init + n_iterations
d = 2  # Dimensionality

# Initialize storage
X_observed = torch.empty((0, d), dtype=torch.float64)
Y_observed = torch.empty((0, 1), dtype=torch.float64)

# Generate initial random samples (Sobol)
sobol = torch.quasirandom.SobolEngine(dimension=d, scramble=True, seed=42)
X_init = sobol.draw(n=n_init).to(dtype=torch.float64)
# Scale to actual bounds
X_init = bounds_lower + (bounds_upper - bounds_lower) * X_init
Y_init = branin_tensor(X_init).unsqueeze(-1)

X_observed = torch.cat([X_observed, X_init])
Y_observed = torch.cat([Y_observed, Y_init])

print(f"Initial best value: {Y_observed.max().item():.4f} (Branin: {-Y_observed.max().item():.4f})")
print(f"Initial best point: {X_observed[Y_observed.argmax()]}")

In [ ]:
# BO Loop
best_values = [Y_observed.max().item()]

for i in range(n_iterations):
    print(f"\nIteration {i+1}/{n_iterations}")
    
    # Fit model
    model = initialize_model(X_observed, standardize(Y_observed))
    fit_model(model)
    model.eval()
    
    # Define acquisition function (Expected Improvement)
    acqf = LogExpectedImprovement(model, standardize(Y_observed).max())
    
    # Optimize acquisition function
    candidate, acq_value = optimize_acqf(
        acqf,
        bounds=bounds,
        q=1,
        num_restarts=10,
        raw_samples=512,
    )
    
    # Evaluate the candidate
    Y_new = branin_tensor(candidate).unsqueeze(-1)
    
    # Update observations
    X_observed = torch.cat([X_observed, candidate])
    Y_observed = torch.cat([Y_observed, Y_new])
    
    best_values.append(Y_observed.max().item())
    
    print(f"  Candidate: {candidate.squeeze().tolist()}")
    print(f"  Value: {Y_new.item():.4f} (Branin: {-Y_new.item():.4f})")
    print(f"  Best so far: {Y_observed.max().item():.4f} (Branin: {-Y_observed.max().item():.4f})")

print("\n" + "="*50)
print("Optimization Complete!")
print(f"Best value found: {Y_observed.max().item():.4f} (Branin: {-Y_observed.max().item():.4f})")
print(f"Best point found: {X_observed[Y_observed.argmax()].tolist()}")
print(f"Known global minimum of Branin: ~0.397887")

## Visualize the Results

In [ ]:
# Plot convergence
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(range(len(best_values)), [-y for y in best_values], 'b-o', linewidth=2, markersize=6)
plt.axhline(y=0.397887, color='r', linestyle='--', label='Global minimum')
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Best Branin Value', fontsize=12)
plt.title('Convergence Plot', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot sampled points
plt.subplot(1, 2, 2)
X_np = X_observed.numpy()
Y_np = -Y_observed.numpy()  # Negate back to minimization

# Create contour plot of Branin function
x1_grid = np.linspace(-5, 10, 100)
x2_grid = np.linspace(0, 10, 100)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
Z = np.zeros_like(X1)
for i in range(X1.shape[0]):
    for j in range(X1.shape[1]):
        Z[i, j] = branin(X1[i, j], X2[i, j])

plt.contourf(X1, X2, Z, levels=20, cmap='viridis', alpha=0.6)
plt.colorbar(label='Branin Value')

# Plot initial points
plt.scatter(X_np[:n_init, 0], X_np[:n_init, 1], c='red', s=100, 
            marker='o', edgecolors='black', linewidth=1.5, label='Initial samples', zorder=5)

# Plot BO points
plt.scatter(X_np[n_init:, 0], X_np[n_init:, 1], c='blue', s=100, 
            marker='s', edgecolors='black', linewidth=1.5, label='BO samples', zorder=5)

# Plot best point
best_idx = Y_observed.argmax()
plt.scatter(X_np[best_idx, 0], X_np[best_idx, 1], c='yellow', s=300, 
            marker='*', edgecolors='black', linewidth=2, label='Best point', zorder=10)

plt.xlabel('x1', fontsize=12)
plt.ylabel('x2', fontsize=12)
plt.title('Sampled Points on Branin Function', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Statistics:")
print(f"  Total evaluations: {len(Y_observed)}")
print(f"  Best Branin value: {-Y_observed.max().item():.6f}")
print(f"  Gap from global minimum: {-Y_observed.max().item() - 0.397887:.6f}")

## Summary

This notebook demonstrates the spherical linear Bayesian optimization method on the 2D Branin function. The key innovations are:

1. **Spherical Projection**: Input points are projected onto a hypersphere using inverse stereographic projection
2. **Linear Kernel**: A simple linear kernel is applied in the projected space
3. **ARD Lengthscales**: Automatic Relevance Determination (ARD) lengthscales allow different scaling per dimension
4. **Global Lengthscale**: An additional global lengthscale parameter controls overall smoothness

The method is computationally efficient (O(ND²) vs O(N³) for standard GPs) while maintaining competitive performance, especially in high-dimensional settings where N ≈ D.